# LoRA fine-tuning for human-preference prediction (LMSYS Chatbot Arena)

Runs the full pipeline from the repo on a Kaggle GPU. **Before running:** Settings -> Accelerator = *GPU T4 x2* (or P100), Internet = *On*.

Order: (1) quick smoke run to prove the GPU path works (about 5 minutes), (2) baselines, (3) the real fine-tune, (4) print metrics.
The data is the public Apache-2.0 Hugging Face copy of the competition training set, so no competition login is needed.

In [ ]:
!git clone https://github.com/SkyWalkersCity/llm-preference-finetuning.git
%cd llm-preference-finetuning
!pip install -q -e . bitsandbytes
!mkdir -p data && curl -L -o data/train.csv https://huggingface.co/datasets/lmarena-ai/arena-human-preference-55k/resolve/main/train.csv
!nvidia-smi

## 1. Smoke run (checks GPU, memory and the whole train + eval path)

In [ ]:
!python -m lmsys_pref.train --data data/train.csv --preset qwen1.5b --output outputs/smoke \
    --max_train_samples 256 --max_val_samples 128 --train_eval_samples 64 \
    --max_steps 8 --eval_steps 4 --save_steps 4 --logging_steps 1

## 2. Non-LLM baselines on the same split

In [ ]:
!python scripts/run_baselines.py --data data/train.csv --out results/baselines.json

## 3. The real fine-tune

`--max_train_samples 20000` keeps one epoch inside a Kaggle session on a T4 (my estimate: about 3 to 5 hours for the 1.5B model; unverified, check the speed in the first log lines and lower the sample count if needed).
For a faster run use `--preset qwen0.5b`. If you hit CUDA out-of-memory, lower `--max_len` (for example 768) or `--batch_size`.

In [ ]:
!python -m lmsys_pref.train --data data/train.csv --preset qwen1.5b --output outputs/qwen1.5b \
    --max_train_samples 20000 --epochs 1 --eval_steps 100 --save_steps 100 --train_eval_samples 1000

## 4. Results (copy this output back)

In [ ]:
import json
print(json.dumps(json.load(open("outputs/qwen1.5b/metrics.json")), indent=2))
print(json.dumps(json.load(open("results/baselines.json")), indent=2))

In [ ]:
# Keep the artifacts: adapter (small) + metrics + validation predictions
!mkdir -p /kaggle/working/results_export
!cp -r outputs/qwen1.5b/adapter outputs/qwen1.5b/metrics.json outputs/qwen1.5b/val_predictions.csv results/baselines.json /kaggle/working/results_export/